# 🗣️ Build a translator: LSTM encoder-decoder

Welcome to the **encdec** challenge! Today you build a **sequence-to-sequence** model from LSTMs (Sutskever, Vinyals & Le, 2014; Cho et al., 2014):
one LSTM **reads** the whole input, squeezes it into its final state `(h, c)`, and a second LSTM **writes** the output from that state, one token at a time.

**The plan** 🗺️
1. Load a dataset: real **English → Hinglish** sentences 🇮🇳, or the offline **dates** task (`"friday, 16 october 2026" → "2026-10-16"`).
2. Turn text into token ids (provided).
3. ✍️ **TODO 1** `Encoder.forward`: embed, pack, run the LSTM, return the final `(h, c)`, the **context**.
4. ✍️ **TODO 2** `Decoder.step`: one LSTM step from the previous token → scores for the next token.
5. ✍️ **TODO 3** teacher forcing: what the decoder reads while training.
6. ✍️ **TODO 4** `greedy_decode`: translating for real, with no answers to peek at.
7. ✍️ **TODO 5** `masked_loss`: cross-entropy that ignores `<pad>`.
8. Train, then score it: exact match (dates) or BLEU-4 (Hinglish).
9. 🍾 **The bottleneck**: does the model get worse on longer inputs?
10. 👩‍🏫 **Teacher forcing vs free-running**: train a second model without the teacher and compare.
11. Translate your own sentence, peek at what's next (attention 👀), then 🚀 submit.

**How to use this notebook**
- `Runtime → Change runtime type → T4 GPU`, then fill in the ⚙️ settings cell and `Runtime → Run all`.
- Fill in every cell marked `# TODO` (replace the `...`). The cell right after each one checks your work ✅.
- `QUICK = True` runs a tiny version in ~3 minutes on a CPU (good for debugging; the scores will be low).

In [ ]:
#@title ⚙️ Settings: paste your platform details, pick a dataset
PLATFORM_URL = ""   #@param {type:"string"}
SUBMIT_TOKEN = ""   #@param {type:"string"}
DATASET = "hinglish"  #@param ["hinglish", "dates"]
QUICK = False  #@param {type:"boolean"}

In [ ]:
#@title 🧰 Imports, seed, device
import math, os, random, re, subprocess, sys, time
from collections import Counter
from datetime import date, timedelta

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Apple's MPS is skipped on purpose: the step-by-step decoder runs ~10x slower there than on the CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  device: {device}   |   dataset: {DATASET}   |   QUICK: {QUICK}")
if device.type != "cuda":
    print("⚠️  No GPU found. Training will be slow. In Colab: Runtime → Change runtime type → T4 GPU "
          "(or set QUICK = True to run a mini version).")

def check(condition, hint):
    """Tiny helper for the ✅ check cells."""
    if not condition:
        raise AssertionError("❌ " + hint)

## 1️⃣ The data

| `DATASET` | input → output | level | notes |
|---|---|---|---|
| `"hinglish"` | `"set an alarm for 7 am"` → `"7 baje ka alarm set karo"` | words | [findnitai/english-to-hinglish](https://huggingface.co/datasets/findnitai/english-to-hinglish) (Apache-2.0), ~189k pairs of chatbot-style commands & questions. We keep short pairs (≤ 12 tokens per side), lowercase, drop duplicate English sentences, and hold out a fixed **2,000-pair test set**. |
| `"dates"` | `"friday, 16 october 2026"` → `"2026-10-16"` | characters | Generated right here, no download. 8 messy formats, including the sneaky month-first `10.16.2026`. Guaranteed to work offline. |

Both datasets end up in the same shape: a list of `(source_text, target_text)` pairs. From there on, **one model and one training loop handle both**.

In [ ]:
#@title 📦 Load the pairs
FORMATS = [
    lambda d: d.strftime("%d %B %Y"),                   # 16 october 2026
    lambda d: d.strftime("%B %d, %Y"),                  # october 16, 2026
    lambda d: d.strftime("%A, %d %B %Y"),               # friday, 16 october 2026
    lambda d: d.strftime("%a %b %d %Y"),                # fri oct 16 2026
    lambda d: f"{d.day} {d.strftime('%b')} {d.year}",   # 16 oct 2026
    lambda d: d.strftime("%d/%m/%Y"),                   # 16/10/2026  (day first)
    lambda d: d.strftime("%m.%d.%Y"),                   # 10.16.2026  (month first!)
    lambda d: f"{d.strftime('%B')} {d.day} {d.year}",   # october 16 2026
]

def make_date_pairs(n, seed):
    rng = random.Random(seed)
    pairs = []
    for _ in range(n):
        d = date(1950, 1, 1) + timedelta(days=rng.randrange(365 * 100))
        pairs.append((rng.choice(FORMATS)(d).lower(), d.isoformat()))
    return pairs

def word_tokenize(text):
    """lowercase, split into words and single punctuation marks: "What's up?" → what ' s up ?"""
    return re.findall(r"[a-z0-9]+|[^\sa-z0-9]", text.lower())

MAX_WORDS = 12
N_TEST = 2000

if DATASET == "dates":
    train_pairs = make_date_pairs(4000 if QUICK else 20000, seed=0)
    test_pairs = make_date_pairs(500 if QUICK else N_TEST, seed=123)
else:
    try:
        import datasets
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=True)
    from datasets import load_dataset
    raw = load_dataset("findnitai/english-to-hinglish", split="train")   # one column: "translation" = {"en", "hi_ng", "source"}
    pairs, seen = [], set()
    for row in raw["translation"]:
        en, hi = (row["en"] or "").strip(), (row["hi_ng"] or "").strip()
        en_tok, hi_tok = word_tokenize(en), word_tokenize(hi)
        if not (0 < len(en_tok) <= MAX_WORDS and 0 < len(hi_tok) <= MAX_WORDS):
            continue
        key = " ".join(en_tok)
        if key in seen:              # same English sentence twice → keep one, so the test set never leaks into training
            continue
        seen.add(key)
        pairs.append((" ".join(en_tok), " ".join(hi_tok)))
    random.Random(42).shuffle(pairs)  # fixed shuffle → everyone gets the SAME test set
    test_pairs, train_pairs = pairs[:N_TEST], pairs[N_TEST:]
    if QUICK:
        train_pairs, test_pairs = train_pairs[:15000], test_pairs[:500]
    print(f"kept {len(pairs):,} short pairs out of {len(raw):,}")

print(f"train: {len(train_pairs):,} pairs   test: {len(test_pairs):,} pairs\n")
for src, tgt in random.Random(1).sample(train_pairs, 8):
    print(f"  {src:42s} →  {tgt}")

## 2️⃣ Tokens → ids (provided)

Neural nets eat numbers, not strings. We split each sentence into **tokens** (characters for dates, words for Hinglish) and give every token an id.
Four special tokens:

| token | id | job |
|---|---|---|
| `<pad>` | 0 | filler so sentences in a batch have the same length (the model must **ignore** it) |
| `<sos>` | 1 | "start of sentence": the first input to the decoder |
| `<eos>` | 2 | "end of sentence": the decoder says this when it's done, so outputs can have **any length** |
| `<unk>` | 3 | any word that's not in the vocabulary (we keep the 8,000 most common words per side) |

A batch is `x` (B, T_in) source ids, `lengths` (B,) the real length of each source, and `y` (B, T_out) target ids ending in `<eos>`, padded with `<pad>`.

In [ ]:
#@title 🔤 Vocab + DataLoaders
PAD, SOS, EOS, UNK = 0, 1, 2, 3
SPECIALS = ["<pad>", "<sos>", "<eos>", "<unk>"]
CHAR_LEVEL = DATASET == "dates"

def tokenize(text):
    return list(text) if CHAR_LEVEL else text.split()   # hinglish text is already word_tokenize'd

def detokenize(tokens):
    return ("" if CHAR_LEVEL else " ").join(tokens)

class Vocab:
    def __init__(self, token_lists, max_size=8000):
        counts = Counter(tok for toks in token_lists for tok in toks)
        self.itos = SPECIALS + [tok for tok, _ in counts.most_common(max_size)]
        self.stoi = {tok: i for i, tok in enumerate(self.itos)}
    def __len__(self):
        return len(self.itos)
    def encode(self, tokens):
        return [self.stoi.get(tok, UNK) for tok in tokens]
    def decode(self, ids):
        out = []
        for i in ids:
            if i == EOS: break
            if i not in (PAD, SOS): out.append(self.itos[i])
        return out

train_tok = [(tokenize(s), tokenize(t)) for s, t in train_pairs]
test_tok = [(tokenize(s), tokenize(t)) for s, t in test_pairs]

def collate(batch):
    """batch: list of (src_tokens, tgt_tokens) → x (B, T_in), lengths (B,), y (B, T_out)"""
    src = [src_vocab.encode(s) for s, _ in batch]
    tgt = [tgt_vocab.encode(t) + [EOS] for _, t in batch]
    x = torch.full((len(batch), max(map(len, src))), PAD, dtype=torch.long)
    y = torch.full((len(batch), max(map(len, tgt))), PAD, dtype=torch.long)
    for i, (s, t) in enumerate(zip(src, tgt)):
        x[i, :len(s)] = torch.tensor(s)
        y[i, :len(t)] = torch.tensor(t)
    return x, torch.tensor([len(s) for s in src]), y

CFG = {   # model size + training budget per dataset ("fr_" = the free-running model in section 12)
    "dates":    dict(emb=32,  hidden=128, dropout=0.0, batch=128, lr=2e-3, epochs=12, fr_epochs=3, minutes=3, fr_minutes=1.5, vocab=100),
    "hinglish": dict(emb=128, hidden=256, dropout=0.2, batch=128, lr=1e-3, epochs=10, fr_epochs=4, minutes=7, fr_minutes=3, vocab=8000),
}[DATASET]
if QUICK:
    CFG.update(epochs=3, fr_epochs=2, minutes=1.0, fr_minutes=0.4)
    if DATASET == "hinglish":
        CFG.update(emb=64, hidden=128, vocab=3000, epochs=12, fr_epochs=4, minutes=1.2)
    else:
        CFG.update(epochs=15, fr_epochs=6)

src_vocab = Vocab([s for s, _ in train_tok], CFG["vocab"])
tgt_vocab = Vocab([t for _, t in train_tok], CFG["vocab"])
MAX_OUT_LEN = max(len(t) for _, t in train_tok) + 1          # +1 for <eos>; cap for greedy decoding

from torch.utils.data import DataLoader
train_loader = DataLoader(train_tok, batch_size=CFG["batch"], shuffle=True, collate_fn=collate)
test_loader = DataLoader(test_tok, batch_size=256, shuffle=False, collate_fn=collate)

unk_rate = sum(i == UNK for s, _ in test_tok for i in src_vocab.encode(s)) / sum(len(s) for s, _ in test_tok)
print(f"source vocab: {len(src_vocab):,}   target vocab: {len(tgt_vocab):,}   max output length: {MAX_OUT_LEN}")
print(f"test-set source tokens that are <unk>: {unk_rate:.1%}")
x, lengths, y = next(iter(train_loader))
print("batch shapes  x:", tuple(x.shape), " lengths:", tuple(lengths.shape), " y:", tuple(y.shape))
print("\nexample:", train_pairs[0])
print("  x ids:", src_vocab.encode(train_tok[0][0]))
print("  y ids:", tgt_vocab.encode(train_tok[0][1]) + [EOS])

## 3️⃣ ✍️ TODO 1: the encoder

The encoder is an **LSTM** that reads the source one token at a time. Quick recap of what's inside each step (that's the `lstm_cell` from the platform):

$$f, i, o = \sigma(\dots) \qquad \tilde c = \tanh(\dots) \qquad c_t = f \odot c_{t-1} + i \odot \tilde c \qquad h_t = o \odot \tanh(c_t)$$

The cell state `c` is updated by **adding**, not by squashing through a matrix every step. As long as the forget gate `f` stays near 1, the gradient flows back along `c` almost unchanged: that's how LSTMs dodge the **vanishing gradient** and remember the start of a long sentence. 🛣️

After the last real token, the final `(h, c)` is the encoder's **context**: the whole sentence summarised in two vectors of size `H`. That's *all* the decoder will ever see of the input.

📦 **Packing.** Sentences in a batch have different lengths, so the short ones are padded with `<pad>`. A plain LSTM would happily keep reading those pads, and its "final" state would be the state *after the padding*. 🙈 `nn.utils.rnn.pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)` tells the LSTM the true length of each sentence, so the `(h, c)` it returns is the state right after each sentence's **last real token**.

Your job in `Encoder.forward`:
1. `packed = ...` pack the embedded batch (`lengths` must live on the CPU, hence `.cpu()`).
2. `out, (h, c) = self.lstm(packed)`: `nn.LSTM` returns all the outputs **and** the final `(h, c)`, each of shape `(num_layers, B, H) = (1, B, H)`.
3. Return `h` and `c` as `(B, H)` each: drop the layer dimension with `h[-1]` (or `h.squeeze(0)`).

⚠️ Don't grab `out[:, -1]` from an unpacked LSTM: for short sentences that's the state after reading padding.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb, hidden, dropout=0.0):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb, padding_idx=PAD)
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(emb, hidden, batch_first=True)   # one layer, one direction

    def forward(self, x, lengths):
        """x: (B, T_in) token ids;  lengths: (B,) real lengths.
        Returns the context (h, c): the LSTM's final state after each sentence's last REAL token, each (B, H)."""
        embedded = self.dropout(self.emb(x))                                 # (B, T_in, emb)
        # TODO 1a: pack the padded batch so the LSTM skips the <pad> tokens
        packed = ...
        # TODO 1b: run the LSTM on the packed batch → out, (h, c)
        out, (h, c) = ...
        # TODO 1c: return the final h and c, each (B, H) (drop the num_layers dimension)
        return ...

In [ ]:
#@title ✅ Check: encoder
torch.manual_seed(0)
_enc = Encoder(vocab_size=11, emb=8, hidden=16).eval()
_x = torch.randint(4, 11, (3, 6)); _x[1, 4:] = PAD; _x[2, 2:] = PAD
_len = torch.tensor([6, 4, 2])
try:
    _ctx = _enc(_x, _len)
except (TypeError, AttributeError, RuntimeError, ValueError) as e:
    raise AssertionError(f"❌ Encoder.forward crashed ({type(e).__name__}: {e}). Fill in TODO 1a, 1b and 1c; the hints are in the markdown above.")
check(isinstance(_ctx, tuple) and len(_ctx) == 2, "Encoder.forward should return TWO tensors: `return h, c` (TODO 1c).")
_h, _c = _ctx
check(isinstance(_h, torch.Tensor) and isinstance(_c, torch.Tensor), "Encoder.forward should return two tensors (h, c).")
check(_h.shape == (3, 16) and _c.shape == (3, 16),
      f"h and c should both be (B, H) = (3, 16), got {tuple(_h.shape)} and {tuple(_c.shape)}. "
      "nn.LSTM gives (num_layers, B, H): use h[-1] and c[-1].")
with torch.no_grad():
    _h2, _c2 = _enc(torch.cat([_x, torch.full((3, 5), PAD)], dim=1), _len)   # same sentences, 5 extra <pad> columns
check(torch.allclose(_h, _h2, atol=1e-6) and torch.allclose(_c, _c2, atol=1e-6),
      "adding extra <pad> at the end changed the context! The LSTM is reading the padding. "
      "Pack the batch with pack_padded_sequence (TODO 1a) and run the LSTM on `packed`.")
with torch.no_grad():   # reference: run each sentence ALONE, without any padding
    _ref = [_enc.lstm(_enc.emb(_x[b:b + 1, :_len[b]]))[1] for b in range(3)]
    _ref_h = torch.cat([r[0][-1] for r in _ref]); _ref_c = torch.cat([r[1][-1] for r in _ref])
check(torch.allclose(_h, _ref_h, atol=1e-5),
      "h is not the state after each sentence's last real token. Use the h that nn.LSTM returns for the PACKED batch "
      "(not out[:, -1], which is the state after the padding).")
check(torch.allclose(_c, _ref_c, atol=1e-5), "h is right but c isn't: return the CELL state c (second part of (h, c)), not h twice.")
print("✅ nice! The encoder ignores padding and hands over the true final (h, c).")

## 4️⃣ ✍️ TODO 2: one decoder step

The decoder is a second LSTM, but we run it **one step at a time** with an `nn.LSTMCell`, because each output token becomes the next input.
It starts from the encoder's context: `(h, c) = encoder(x)`. Then every step:

1. Embed the previous token: `self.emb(prev_token)` → `(B, emb)` (wrap it in `self.dropout(...)`).
2. One LSTM step: `h, c = self.cell(emb, (h, c))`. Note the state goes in as a **tuple**.
3. Project the **new** `h` to one score per target word: `self.out(h)` → logits `(B, V)`.

Return `logits, h, c` so the next step can continue from the new state.

In [ ]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, emb, hidden, dropout=0.0):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb, padding_idx=PAD)
        self.dropout = nn.Dropout(dropout)
        self.cell = nn.LSTMCell(emb, hidden)
        self.out = nn.Linear(hidden, vocab_size)

    def step(self, prev_token, h, c):
        """prev_token: (B,) ids;  h, c: (B, H).  Returns logits (B, V) and the new h, c."""
        # TODO 2a: embed the previous token (and apply dropout) → (B, emb)
        emb = ...
        # TODO 2b: one LSTMCell step from the state (h, c)
        h, c = ...
        # TODO 2c: project the new h to scores over the target vocab → (B, V)
        logits = ...
        return logits, h, c

In [ ]:
#@title ✅ Check: decoder step
torch.manual_seed(0)
_dec = Decoder(vocab_size=7, emb=8, hidden=16).eval()
_tok = torch.tensor([SOS, 4, 5]); _h0, _c0 = torch.randn(3, 16), torch.randn(3, 16)
try:
    _res = _dec.step(_tok, _h0, _c0)
    _logits, _h1, _c1 = _res
except (TypeError, AttributeError, RuntimeError, ValueError) as e:
    raise AssertionError(f"❌ Decoder.step crashed ({type(e).__name__}: {e}). Fill in TODO 2a, 2b and 2c.")
check(all(isinstance(t, torch.Tensor) for t in (_logits, _h1, _c1)), "Decoder.step should return three tensors: logits, h, c.")
check(_logits.shape == (3, 7), f"logits should be (B, V) = (3, 7), got {tuple(_logits.shape)}.")
check(_h1.shape == (3, 16) and _c1.shape == (3, 16), f"h, c should be (B, H) = (3, 16), got {tuple(_h1.shape)}, {tuple(_c1.shape)}.")
with torch.no_grad():
    _rh, _rc = _dec.cell(_dec.emb(_tok), (_h0, _c0))
check(torch.allclose(_h1, _rh, atol=1e-6) and torch.allclose(_c1, _rc, atol=1e-6),
      "the new (h, c) is off. Start from the GIVEN state: self.cell(emb, (h, c)), with emb = self.emb(prev_token).")
check(torch.allclose(_logits, _dec.out(_rh), atol=1e-6), "logits should be self.out(new h): project the state AFTER the step.")
print("✅ nice! One decoder step: embed → LSTMCell → linear.")

## 5️⃣ ✍️ TODO 3: teacher forcing

Now glue encoder and decoder together. While **training** we know the right answer `target`, so at every step we feed the decoder the **true** previous token `target[:, t]`, even if the model guessed wrong. That's **teacher forcing** 👩‍🏫: training is fast and stable, because one early mistake can't derail the rest of the sentence.

```
step:     0       1       2       3
input:  <sos>   y[0]    y[1]    y[2]     ← the TRUE tokens (teacher forcing)
predict: y[0]   y[1]    y[2]    y[3]
```

The `teacher_forcing=False` branch (**free-running**: feed the model's own guess, `argmax`) is provided. We'll use it in section 1️⃣2️⃣ to see what happens without the teacher.

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, src_size, tgt_size, emb=32, hidden=64, dropout=0.0):
        super().__init__()
        self.encoder = Encoder(src_size, emb, hidden, dropout)
        self.decoder = Decoder(tgt_size, emb, hidden, dropout)

    def forward(self, x, lengths, target, teacher_forcing=True):
        """Training forward pass. target: (B, T_out). Returns logits (B, T_out, V): step t predicts target[:, t]."""
        h, c = self.encoder(x, lengths)                                    # the context starts the decoder
        token = torch.full((x.size(0),), SOS, dtype=torch.long, device=x.device)
        logits = []
        for t in range(target.size(1)):
            step_logits, h, c = self.decoder.step(token, h, c)
            logits.append(step_logits)
            if teacher_forcing:
                # TODO 3: the next input is the TRUE token at position t
                token = ...
            else:
                token = step_logits.argmax(dim=1)                          # free-running: the model's own guess
        return torch.stack(logits, dim=1)

In [ ]:
#@title ✅ Check: teacher forcing
torch.manual_seed(0)
_m = Seq2Seq(src_size=11, tgt_size=7, emb=8, hidden=16).eval()
_y = torch.randint(4, 7, (3, 5))
try:
    with torch.no_grad():
        _logits = _m(_x, _len, target=_y)
except (TypeError, AttributeError, RuntimeError, IndexError, ValueError) as e:
    raise AssertionError(f"❌ Seq2Seq crashed ({type(e).__name__}: {e}). Fill in TODO 3 (the encoder and decoder checks must pass first).")
check(isinstance(_logits, torch.Tensor) and _logits.shape == (3, 5, 7), "logits should be (B, T_out, V) = (3, 5, 7).")
_y2 = _y.clone(); _y2[:, 2] = 4 + (_y[:, 2] - 4 + 1) % 3     # change ONLY the true token at position 2
with torch.no_grad():
    _logits2 = _m(_x, _len, target=_y2)
check(torch.allclose(_logits[:, :3], _logits2[:, :3]),
      "steps 0-2 changed when only target[:, 2] changed: the decoder is peeking ahead. After step t, feed target[:, t] (not t+1).")
check(not torch.allclose(_logits[:, 3], _logits2[:, 3]),
      "the true tokens are ignored! With teacher forcing the input at step t+1 must be target[:, t], "
      "not the model's own argmax guess.")
with torch.no_grad():   # reference: walk the decoder by hand with the true tokens
    _h, _c = _m.encoder(_x, _len); _tok = torch.full((3,), SOS); _ref = []
    for t in range(5):
        _l, _h, _c = _m.decoder.step(_tok, _h, _c); _ref.append(_l); _tok = _y[:, t]
check(torch.allclose(_logits, torch.stack(_ref, 1), atol=1e-5), "logits don't match: start from <sos>, then feed target[:, t] after step t.")
print("✅ nice! The decoder reads the true previous token while training.")

## 6️⃣ ✍️ TODO 4: greedy decoding

At test time there's no `target`, so the decoder has to feed itself:

1. Encode the source → `(h, c)`.
2. `token` starts as `<sos>` for every row: `torch.full((B,), SOS, dtype=torch.long, device=x.device)`.
3. Each step: `step_logits, h, c = model.decoder.step(token, h, c)`, then pick the most likely word: `token = step_logits.argmax(dim=1)`.
4. Remember `token`. Stop when **every** row has said `<eos>` at least once (`finished.all()`), or after `max_len` steps.

It's called **greedy** because it always takes the single best word right now, never reconsidering (beam search keeps the top-k candidates instead).

In [ ]:
@torch.no_grad()
def greedy_decode(model, x, lengths, max_len):
    """x: (B, T_in), lengths: (B,). Returns predicted ids (B, steps) with steps ≤ max_len."""
    h, c = model.encoder(x, lengths)
    # TODO 4a: every row starts with <sos>, shape (B,)
    token = ...
    finished = torch.zeros(x.size(0), dtype=torch.bool, device=x.device)
    outputs = []
    for _ in range(max_len):
        # TODO 4b: one decoder step, then the most likely next token (B,)
        step_logits, h, c = ...
        token = ...
        outputs.append(token)
        finished |= token == EOS
        # TODO 4c: stop early once every row has produced <eos>
        if ...:
            break
    return torch.stack(outputs, dim=1)

In [ ]:
#@title ✅ Check: greedy decoding
torch.manual_seed(0)
_m = Seq2Seq(src_size=11, tgt_size=7, emb=8, hidden=16).eval()
with torch.no_grad():
    _m.decoder.out.bias[EOS] = -1e4          # this model can never say <eos> → must run all max_len steps
try:
    _g = greedy_decode(_m, _x, _len, max_len=5)
except (TypeError, AttributeError, RuntimeError, ValueError) as e:
    raise AssertionError(f"❌ greedy_decode crashed ({type(e).__name__}: {e}). Fill in TODO 4a, 4b and 4c.")
check(isinstance(_g, torch.Tensor) and _g.dtype == torch.long, "greedy_decode should return a LongTensor of token ids.")
check(_g.shape == (3, 5), f"with no <eos> in sight, greedy_decode should run exactly max_len=5 steps → (3, 5), got {tuple(_g.shape)}. "
      "Only `break` when finished.all() (TODO 4c).")
try:
    with torch.no_grad():
        _tf = _m(_x, _len, target=_g)        # teacher-force the model on its OWN outputs: argmax must reproduce them
except (TypeError, AttributeError, RuntimeError, IndexError, ValueError) as e:
    raise AssertionError(f"❌ Seq2Seq crashed ({type(e).__name__}: {e}). The teacher-forcing check (TODO 3) must pass first.")
check((_tf.argmax(-1) == _g).all(),
      "the outputs aren't the model's own argmax choices. Start from <sos> (TODO 4a), and feed back token = step_logits.argmax(dim=1) (TODO 4b).")
with torch.no_grad():
    _m.decoder.out.bias[EOS] = 1e4           # now the model ALWAYS says <eos> → should stop after 1 step
    _g1 = greedy_decode(_m, _x, _len, max_len=5)
check(_g1.shape == (3, 1) and (_g1 == EOS).all(),
      f"when every row says <eos> at step 1, decoding should stop right away (got {_g1.shape[1]} steps). Use `if finished.all(): break`.")
from types import SimpleNamespace
def _stub_step(tok, h, c):   # row 0 always says <eos>, the other rows never do
    logits = torch.zeros(3, 7); logits[:, 4] = 1.0; logits[:, EOS] = 10.0 * h[:, 0]
    return logits, h, c
_stub = SimpleNamespace(encoder=lambda x, lengths: (torch.tensor([[1.0], [0.0], [0.0]]), torch.zeros(3, 1)),
                        decoder=SimpleNamespace(step=_stub_step))
_g2 = greedy_decode(_stub, _x, _len, max_len=5)
check(_g2.shape == (3, 5), f"only row 0 is done after step 1, but decoding stopped after {_g2.shape[1]} step(s). "
      "Keep going until ALL rows have said <eos>: finished.all(), not finished.any().")
print("✅ nice! Greedy decoding: start at <sos>, feed back the argmax, stop at <eos> or max_len.")

## 7️⃣ ✍️ TODO 5: the masked loss

The decoder makes a prediction at every position of `y`, including the `<pad>` positions after `<eos>`. Those are **not real words**: if they count in the loss, the model learns to predict padding (easy points 🥱) and short sentences get a weird extra weight.

Loss = cross-entropy averaged over the **real** target tokens only:
- the easy way: `F.cross_entropy(logits.reshape(-1, V), target.reshape(-1), ignore_index=PAD)`. `cross_entropy` wants `(N, V)` and `(N,)`, hence the reshapes.
- by hand: per-token loss with `reduction="none"`, multiply by `mask = target != PAD`, then divide by `mask.sum()` (**not** by B·T).

In [ ]:
def masked_loss(logits, target):
    """logits: (B, T, V);  target: (B, T) ids with PAD after <eos>.
    Returns ONE number: the mean cross-entropy over the non-pad positions (keep it a tensor so .backward() works)."""
    # TODO 5: cross-entropy that ignores <pad>
    return ...

In [ ]:
#@title ✅ Check: masked loss
torch.manual_seed(0)
_lg = torch.randn(2, 4, 7, requires_grad=True)
_t = torch.tensor([[4, 5, EOS, PAD], [6, EOS, PAD, PAD]])
try:
    _loss = masked_loss(_lg, _t)
except (TypeError, AttributeError, RuntimeError, ValueError) as e:
    raise AssertionError(f"❌ masked_loss crashed ({type(e).__name__}: {e}). Fill in TODO 5; remember the reshapes to (N, V) and (N,).")
check(isinstance(_loss, torch.Tensor) and _loss.dim() == 0, "masked_loss should return ONE number as a 0-d tensor. Did you fill in TODO 5? (And don't call .item(): keep it a tensor.)")
check(_loss.requires_grad, "the loss must keep its gradient (don't call .item() or .detach()).")
_tok_loss = -_lg.log_softmax(-1).gather(-1, _t.unsqueeze(-1)).squeeze(-1).detach()   # (2, 4) loss of every position
_real = _t != PAD
_ref = _tok_loss[_real].mean()
check(not torch.isclose(_loss, _tok_loss.mean()), "you're counting the <pad> positions! Use ignore_index=PAD (or a mask).")
check(not torch.isclose(_loss, _tok_loss[_real].sum() / _t.numel()), "the padding is masked, but you divide by B·T. Divide by the number of REAL tokens: mask.sum().")
check(not torch.isclose(_loss, _tok_loss[_real].sum()), "that's the SUM over tokens; we want the mean over the real tokens.")
check(torch.isclose(_loss, _ref, atol=1e-5), f"expected the mean cross-entropy over the 5 real tokens ({_ref:.4f}), got {_loss.item():.4f}.")
_lg_pad = torch.cat([_lg.detach(), 100 * torch.randn(2, 3, 7)], dim=1); _t_pad = torch.cat([_t, torch.full((2, 3), PAD)], dim=1)
check(torch.isclose(masked_loss(_lg_pad, _t_pad), _ref, atol=1e-5), "extra padding changed the loss: <pad> positions must not count.")
print("✅ nice! The loss only counts the real tokens.")

## 8️⃣ How we score translations 📏

The loss tells us how surprised the model is by the right answer. But what we actually care about is the translation it **writes** with `greedy_decode`. So we score that:

- **dates** → **exact match**: % of test dates where the whole output string is exactly right. One wrong digit = wrong.
- **hinglish** → **BLEU-4**, the classic machine-translation score (0–100). For every n = 1…4 it counts how many of the model's n-grams also appear in the reference (clipped, so repeating "hai hai hai" doesn't farm points), takes the geometric mean of those 4 precisions, and multiplies by a **brevity penalty** `exp(1 − ref_len / hyp_len)` when the output is too short (otherwise a 1-word answer would score great precision). Computed over the whole corpus at once, in a few lines of plain Python below 👇

In [ ]:
#@title 🧮 Scoring helpers (provided)
def translate_pairs(model, tok_pairs, batch_size=256):
    """greedy-decode every source → list of output token lists"""
    model.eval()
    outs = []
    for i in range(0, len(tok_pairs), batch_size):
        x, lengths, _ = collate(tok_pairs[i:i + batch_size])
        ids = greedy_decode(model, x.to(device), lengths, max_len=MAX_OUT_LEN).cpu().tolist()
        outs += [tgt_vocab.decode(row) for row in ids]
    model.train()
    return outs

def bleu_stats(h, r, max_n=4):
    """clipped n-gram matches and totals for one sentence pair"""
    matches, totals = [], []
    for n in range(1, max_n + 1):
        h_ngrams = Counter(tuple(h[i:i + n]) for i in range(len(h) - n + 1))
        r_ngrams = Counter(tuple(r[i:i + n]) for i in range(len(r) - n + 1))
        matches.append(sum(min(c, r_ngrams[g]) for g, c in h_ngrams.items()))
        totals.append(max(len(h) - n + 1, 0))
    return matches, totals

def bleu_from_stats(matches, totals, hyp_len, ref_len, smooth=0):
    if matches[0] == 0 or (min(matches) == 0 and not smooth):
        return 0.0
    log_precision = sum(math.log((m + (smooth if n else 0)) / max(t + (smooth if n else 0), 1))
                        for n, (m, t) in enumerate(zip(matches, totals))) / len(matches)
    brevity_penalty = 1.0 if hyp_len > ref_len else math.exp(1 - ref_len / max(hyp_len, 1))
    return 100 * brevity_penalty * math.exp(log_precision)

def corpus_bleu(hyps, refs, max_n=4):
    """hyps, refs: lists of token lists. Returns BLEU-4 in [0, 100]."""
    matches, totals = [0] * max_n, [0] * max_n
    for h, r in zip(hyps, refs):
        m, t = bleu_stats(h, r, max_n)
        matches = [a + b for a, b in zip(matches, m)]; totals = [a + b for a, b in zip(totals, t)]
    return bleu_from_stats(matches, totals, sum(map(len, hyps)), sum(map(len, refs)))

def sentence_bleu(h, r):
    """BLEU-4 for ONE sentence, with add-1 smoothing on n ≥ 2 (else almost every sentence scores 0)"""
    if not h:
        return 0.0
    m, t = bleu_stats(h, r)
    return bleu_from_stats(m, t, len(h), len(r), smooth=1)

assert abs(corpus_bleu([["a", "b", "c", "d"]], [["a", "b", "c", "d"]]) - 100) < 1e-9   # perfect = 100

refs = [t for _, t in test_tok]
METRIC_NAME = "exact_match" if DATASET == "dates" else "bleu"

def test_score(model):
    """greedy-translate the whole test set → (score, predictions)"""
    preds = translate_pairs(model, test_tok)
    if DATASET == "dates":
        return 100 * sum(h == r for h, r in zip(preds, refs)) / len(refs), preds
    return corpus_bleu(preds, refs), preds

print(f"metric for {DATASET}: {METRIC_NAME}")

## 9️⃣ Train it 🏋️

Standard loop: teacher-forced forward pass → `masked_loss` → backprop → **gradient clipping** (`clip_grad_norm_`: LSTMs fixed *vanishing* gradients, but gradients can still occasionally *explode*; clipping caps their size) → Adam step.

After every epoch we check the test set three ways:
- **test loss, teacher-forced**: the decoder reads the true previous tokens (same conditions as training).
- **test loss, free-running**: the decoder reads its own guesses, like when it translates for real.
- **test score**: the real translations from `greedy_decode`, scored with exact match / BLEU.

⏱️ The model trains for its epochs **or** its time budget, whichever runs out first (~7 min on a T4 for Hinglish with `QUICK = False`).

In [ ]:
def eval_losses(model, loader):
    """test loss per real token: teacher-forced and free-running"""
    model.eval()
    totals, n = {"tf": 0.0, "free": 0.0}, 0
    with torch.no_grad():
        for x, lengths, y in loader:
            x, y = x.to(device), y.to(device)
            k = (y != PAD).sum().item()
            totals["tf"] += masked_loss(model(x, lengths, y, teacher_forcing=True), y).item() * k
            totals["free"] += masked_loss(model(x, lengths, y, teacher_forcing=False), y).item() * k
            n += k
    model.train()
    return totals["tf"] / n, totals["free"] / n

def train_model(teacher_forcing, epochs, minutes):
    name = "teacher forcing" if teacher_forcing else "free-running"
    torch.manual_seed(SEED)
    model = Seq2Seq(len(src_vocab), len(tgt_vocab), CFG["emb"], CFG["hidden"], CFG["dropout"]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=CFG["lr"])
    hist = {"train": [], "test_tf": [], "test_free": [], "score": []}
    start = time.time()
    for epoch in range(1, epochs + 1):
        total, n = 0.0, 0
        for x, lengths, y in train_loader:
            x, y = x.to(device), y.to(device)
            loss = masked_loss(model(x, lengths, y, teacher_forcing=teacher_forcing), y)
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item(); n += 1
            if time.time() - start > minutes * 60:     # budget is checked every batch
                break
        test_tf, test_free = eval_losses(model, test_loader)
        for key, val in [("train", total / n), ("test_tf", test_tf), ("test_free", test_free), ("score", test_score(model)[0])]:
            hist[key].append(val)
        elapsed = time.time() - start
        print(f"  {name:15s} epoch {epoch:2d}  train loss {hist['train'][-1]:.3f} | test loss: teacher-forced {test_tf:.3f}, "
              f"free-running {test_free:.3f} | test {METRIC_NAME} {hist['score'][-1]:5.1f}  ({elapsed:.0f}s)")
        if elapsed > minutes * 60 and epoch < epochs:
            print(f"  ⏱️ time budget of {minutes} min used up, stopping early")
            break
    return model, hist

n_params = sum(p.numel() for p in Seq2Seq(len(src_vocab), len(tgt_vocab), CFG["emb"], CFG["hidden"]).parameters())
print(f"🏋️ training the encoder-decoder with teacher forcing ({n_params:,} params)")
model, hist_tf = train_model(True, CFG["epochs"], CFG["minutes"])

## 🔟 How good is it? 🎯

🤔 **What to expect.**
- **Dates**: this one is learnable to perfection: expect **99–100%** exact match with `QUICK = False`, and ~90%+ even in QUICK mode on a CPU.
- **Hinglish**: a small model trained for a few minutes, with only **one** reference translation per sentence (and Hinglish spelling is wild: *nahi / nahin / nhi*). In our test run it reached **BLEU ≈ 32** after just 2 epochs, so expect the **30s**. It nails short, common commands (*"cancel 1 hour alarm" → "1 hour ke alarm ko cancel kardo"*) but loses details in longer ones: names and numbers get swapped for something common or become `<unk>`. As a sanity check we also score a **trivial baseline**: just copy the English input. Hinglish borrows lots of English words (alarm, traffic, play…) so copying scores ≈ 8, not 0. With `QUICK = True` expect a BLEU in the low teens.

In [ ]:
METRIC, preds = test_score(model)
if DATASET == "dates":
    per_example = [100.0 * (h == r) for h, r in zip(preds, refs)]     # one score per test example, for the bottleneck plot
    print(f"🎯 exact match: {METRIC:.1f}%")
else:
    per_example = [sentence_bleu(h, r) for h, r in zip(preds, refs)]
    copy_score = corpus_bleu([s for s, _ in test_tok], refs)
    print(f"🎯 BLEU-4: {METRIC:.1f}    copy-the-English baseline: {copy_score:.1f}")

print("\n10 test translations:")
for i in range(10):
    src, ref = test_pairs[i]
    mark = "✅" if preds[i] == refs[i] else "  "
    print(f"{mark} {src}\n     model: {detokenize(preds[i])}\n     ref:   {ref}")

## 1️⃣1️⃣ The bottleneck 🍾

Here's the catch with this design: **every** input, 2 words or 12, has to squeeze through the same fixed-size `(h, c)` (`2 × hidden` numbers). The decoder never sees the input again. So the longer the input, the more has to be crammed into that one state, and the more details get lost (names, numbers, the *third* thing in a list…).

Let's check: score every test example on its own (exact match for dates, a smoothed per-sentence BLEU for Hinglish), group them by **input length**, and plot.

How to read it:
- 📉 **Hinglish**: the score drops as the English sentence gets longer. Part of that is simply that long sentences are harder for anyone, but the bottleneck makes it worse: look at the long examples above, the translation is often fluent but **wrong** (the model "forgot" a detail and filled in something common).
- 📅 **Dates**: the inputs are short and the output is always 10 characters, so `(h, c)` has plenty of room: the curve is much flatter. Here the *format* matters more than the length (the shortest inputs are the all-digit ones like `04/07/2010` vs `04.07.2010`, where day and month swap places).

In [ ]:
#@title 📏 Score vs input length
src_len = np.array([len(s) for s, _ in test_tok])
scores = np.array(per_example)
edges = np.unique(np.quantile(src_len, np.linspace(0, 1, 7)).round()).astype(int)   # ~6 bins with similar counts
bins = [(lo, hi) for lo, hi in zip(edges[:-1], edges[1:])]
bins = [(lo, hi if i == len(bins) - 1 else hi - 1) for i, (lo, hi) in enumerate(bins)]  # inclusive, non-overlapping
len_labels, len_scores, len_counts = [], [], []
for lo, hi in bins:
    sel = (src_len >= lo) & (src_len <= hi)
    if sel.sum() >= 10:
        len_labels.append(f"{lo}" if lo == hi else f"{lo}–{hi}")
        len_scores.append(scores[sel].mean()); len_counts.append(int(sel.sum()))
unit = "characters" if CHAR_LEVEL else "words"
score_label = "exact match %" if DATASET == "dates" else "sentence BLEU (smoothed)"

def plot_bottleneck(ax):
    ax.plot(range(len(len_scores)), len_scores, "-o", color="tab:purple")
    for i, (s, n) in enumerate(zip(len_scores, len_counts)):
        ax.annotate(f"n={n}", (i, s), textcoords="offset points", xytext=(0, 7), ha="center", fontsize=8, color="gray")
    ax.set_xticks(range(len(len_labels))); ax.set_xticklabels(len_labels)
    ax.set_xlabel(f"input length ({unit})"); ax.set_ylabel(score_label)
    ax.set_ylim(0, max(len_scores) * 1.2 + 1)
    ax.set_title("the bottleneck: score vs input length")

fig, ax = plt.subplots(figsize=(7, 4)); plot_bottleneck(ax); plt.tight_layout(); plt.show()
for lab, s, n in zip(len_labels, len_scores, len_counts):
    print(f"  length {lab:>6s}: {score_label} {s:5.1f}   ({n} examples)")

## 1️⃣2️⃣ Teacher forcing 🆚 free-running

What if we train **without** the teacher? Same model, but during training the decoder always reads its **own** previous guess (`teacher_forcing=False`), exactly like at test time. It gets a smaller budget; it's here to show the difference, not to win.

🧠 **Exposure bias.** A teacher-forced model only ever sees **perfect** prefixes during training. At test time it reads its own outputs, and after its first mistake it's in a situation it never practised, so errors can snowball. You can *see* this in the left plot below: for the teacher-forced model (blue), the test loss with its own guesses (dotted) sits well above the test loss with the true prefixes (solid). Same model, same sentences: the only difference is who writes the prefix. That gap is the exposure bias.

The free-running model (orange) has no such gap: it trains exactly the way it's tested. Its loss can even look *better*. So why does everyone still use teacher forcing? 👉 Look at the **middle plot**, the actual translations. Early in training the model's own guesses are garbage, so at step 5 it gets a garbage prefix and is still asked for the true word number 5. It can't learn to *continue* a sentence that way, so it learns to hedge: "what word is usually at position 5?" ➡️ generic, wrong translations. Teacher forcing gives a clean, useful signal at every step from the very first batch. 🌀

On **dates** the task is so regular that the free-running model gets close within a few epochs; the difference there is mostly *speed* (compare the curves at epochs 1–2). On **Hinglish** the gap is huge, and its outputs stutter (*"mere alarm alarm alarm ko ko kare"*): a classic sign of a model that never learned to continue its own sentence.

(Lesson 2: in seq2seq, **loss ≠ quality**. Always look at the real metric. And fixes for exposure bias exist, e.g. *scheduled sampling*: start with the teacher, then slowly mix in the model's own guesses.)

In [ ]:
print("🏋️ training a second model WITHOUT teacher forcing (smaller budget)")
model_free, hist_free = train_model(False, CFG["fr_epochs"], CFG["fr_minutes"])
free_score, preds_free = test_score(model_free)
print(f"\n🎯 test {METRIC_NAME}:  teacher forcing {METRIC:.1f}   free-running {free_score:.1f}")
print("\nsame 3 test sentences, free-running model:")
for i in range(3):
    print(f"   {test_pairs[i][0]}\n     free-running: {detokenize(preds_free[i])}\n     teacher-forced: {detokenize(preds[i])}")

In [ ]:
#@title 📉 Loss curves + score + bottleneck → result.png
def plot_losses(ax):
    ep = range(1, len(hist_tf["train"]) + 1)
    ax.plot(ep, hist_tf["train"], "--x", color="tab:blue", alpha=0.6, label="teacher forcing: train")
    ax.plot(ep, hist_tf["test_tf"], "-o", color="tab:blue", label="teacher forcing: test (true prefixes)")
    ax.plot(ep, hist_tf["test_free"], ":s", color="tab:blue", label="teacher forcing: test (own guesses)")
    ep = range(1, len(hist_free["train"]) + 1)
    ax.plot(ep, hist_free["train"], "--x", color="tab:orange", alpha=0.6, label="free-running: train")
    ax.plot(ep, hist_free["test_free"], ":s", color="tab:orange", label="free-running: test (own guesses)")
    ax.set_xlabel("epoch"); ax.set_ylabel("cross-entropy per token")
    ax.set_title("loss: teacher forcing vs free-running"); ax.legend(fontsize=7)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))

def plot_scores(ax):
    for hist, label, color in [(hist_tf, "teacher forcing", "tab:blue"), (hist_free, "free-running", "tab:orange")]:
        ax.plot(range(1, len(hist["score"]) + 1), hist["score"], "-o", color=color, label=label)
    ax.set_xlabel("epoch"); ax.set_ylabel(f"test {METRIC_NAME}"); ax.set_title("what really matters: the translations")
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax.legend(fontsize=8)

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
plot_losses(axes[0]); plot_scores(axes[1]); plot_bottleneck(axes[2])
fig.suptitle(f"LSTM encoder-decoder on {DATASET} · {METRIC_NAME} = {METRIC:.1f} (free-running model: {free_score:.1f})", fontsize=12)
plt.tight_layout()
plt.savefig("result.png", dpi=100, bbox_inches="tight")
plt.show()
print(f"saved result.png ({os.path.getsize('result.png') / 1e6:.2f} MB)")

## 1️⃣3️⃣ Translate your own sentence 🎤

Type anything in `MY_SENTENCE` (leave it empty for a default). Tips: keep it short, chatbot-ish (alarms, weather, music, traffic, reminders…) because that's what the data is about. Words the model never saw become `<unk>`.
For `dates`, try any of the formats, e.g. `"25 dec 1999"` or `"07.04.1976"`.

In [ ]:
MY_SENTENCE = ""  #@param {type:"string"}

def translate(text):
    src = list(text.lower()) if CHAR_LEVEL else word_tokenize(text)
    src = [tok for tok in src if tok in src_vocab.stoi or not CHAR_LEVEL]   # drop unknown characters for dates
    (out,) = translate_pairs(model, [(src, [])])
    print(f"{text}  →  {detokenize(out)}")
    return detokenize(out)

if DATASET == "dates":
    examples = ["saturday, 1 january 2000", "25 dec 1999", "07.04.1976"]
else:
    examples = ["set an alarm for 7 am tomorrow", "what is the weather like today", "play some music by arijit singh"]
for s in examples[1:]:
    translate(s)
translate(MY_SENTENCE or examples[0]);

## 🔭 What's next: attention

Your bottleneck plot is the whole motivation for the next big idea. Instead of forcing the entire input through one `(h, c)`, what if the decoder could **look back at every encoder state** (`out`, the thing we threw away in TODO 1!) and decide, at each output step, which input words matter right now?

That's **attention** (Bahdanau, Cho & Bengio, 2015). It flattens the score-vs-length curve, makes long sentences work, and if you keep pulling that thread (drop the LSTMs, keep only attention) you end up at the **Transformer**, the architecture behind every modern LLM. 🚀

## 🚀 Submit

Run the cell below. It sends your score (`exact_match` % for dates, `bleu` for Hinglish) and your `result.png` (loss curves, scores per epoch, bottleneck plot) to DL Lab. Make sure `PLATFORM_URL` and `SUBMIT_TOKEN` are filled in at the top.

🧪 **Want a better score?** Try more epochs, a bigger `hidden`, a larger vocab, a 2-layer LSTM, or feeding the source **reversed** (a classic trick from Sutskever et al.: the first source words end up closer to the first output words). Write what you tried in `notes`!

In [ ]:
#@title 🚀 Submit your result to DL Lab
import base64, json, urllib.request, urllib.error
CHALLENGE_ID = "encdec"   # "sequences" | "encdec"

def submit(metric_name=None, metric=None, image_path=None, notes=""):
    if not PLATFORM_URL or not SUBMIT_TOKEN:
        print("⚠️ Paste PLATFORM_URL and SUBMIT_TOKEN into the settings cell at the top (copy them from the platform's challenge page).")
        return
    body = {"dataset": DATASET, "metric_name": metric_name, "metric": None if metric is None else float(metric), "notes": notes}
    if image_path:
        with open(image_path, "rb") as f:
            body["image_b64"] = base64.b64encode(f.read()).decode()
    req = urllib.request.Request(PLATFORM_URL.rstrip("/") + f"/api/challenges/{CHALLENGE_ID}/submit",
                                 data=json.dumps(body).encode(), method="POST",
                                 headers={"Content-Type": "application/json", "Authorization": "Bearer " + SUBMIT_TOKEN.strip()})
    try:
        with urllib.request.urlopen(req, timeout=60) as r:
            print("🎉", json.loads(r.read())["message"])
    except urllib.error.HTTPError as e:
        print("❌ Submit failed:", e.code, e.read().decode()[:300])
    except Exception as e:
        print("❌ Could not reach the platform:", e)

submit(metric_name="exact_match" if DATASET == "dates" else "bleu", metric=METRIC, image_path="result.png",
       notes=f"LSTM encoder-decoder (no attention), hidden={CFG['hidden']}, {len(hist_tf['train'])} epochs, QUICK={QUICK}; "
             f"free-running model scored {free_score:.1f}")